In [0]:
def base_tables(schema):
    """Tables in workspace.<schema>, excluding views - rows cannot be deleted from a view."""
    rows = spark.sql(f"""
        SELECT table_name
        FROM workspace.information_schema.tables
        WHERE table_schema = '{schema}'
          AND table_type NOT IN ('VIEW', 'MATERIALIZED_VIEW')
        ORDER BY table_name
    """).collect()
    return [row.table_name for row in rows]

staging_tables = base_tables("staging")
dw_tables = base_tables("dw")

def delete_all_rows(schema, table):
    spark.sql(f"DELETE FROM {schema}.{table}")

for table in staging_tables:
    delete_all_rows('workspace.staging', table)

for table in dw_tables:
    if table == 'calendar':
        print("Skipping workspace.dw.calendar")
        continue
    delete_all_rows('workspace.dw', table)

print("All rows deleted from staging and dw tables (views and dw.calendar skipped).")